<a href="https://colab.research.google.com/github/estefania-apaza/state-capacity-protest-peru/blob/main/Avance_de_base_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Avance de la Base de Datos
Propuesta de Investigación
- Curso: Estadística para el Análisis Político 2
- Nombres: Estefanía Apaza (20230487) y Diego Luyo (20230934)

**1. Limpieza y filtrado de la base de datos**

In [21]:
import pandas as pd

# Cargado de la base de la Escuela de Gobierno y Políticas Públicas PUCP
file_path = "Base de Eventos de Protesta del Perú_1980_2025.csv"
df_egpp = pd.read_csv(file_path, sep=';', encoding='latin-1', on_bad_lines='skip')

# Limpieza de nombres de columnas
df_egpp.columns = df_egpp.columns.str.strip().str.lower()
df_egpp.columns = df_egpp.columns.str.replace('ñ', 'n').str.replace('á', 'a').str.replace('é', 'e').str.replace('í', 'i').str.replace('ó', 'o').str.replace('ú', 'u')

# Selección preliminar de columnas
columnas_analisis = [
    'id', 'ano', 'mes_id', 'presidente_id',
    'region_id', 'provincia_id', 'distrito_id',
    'sector_id_1', 'actor_1_id', 'sector_id_2', 'actor_2_id',
    'accion_1_id', 'accion_2_id', 'accion_3_id', 'accion_4_id',
    'duracion_horas', 'numero_participantes',
    'numero_heridos', 'numero_muertos', 'numero_detenidos',
    'adversario_id', 'institucion_id',
    'reclamo_id', 'sub_reclamo_id',
    'reclamo_t', 'comentar_t']

# Creación de la base final
df_final = df_egpp[[c for c in columnas_analisis if c in df_egpp.columns]].copy()

df_final.columns
df_final.head(5)

,id,ano,mes_id,presidente_id,region_id,provincia_id,distrito_id,sector_id_1,actor_1_id,sector_id_2,...,numero_participantes,numero_heridos,numero_muertos,numero_detenidos,adversario_id,institucion_id,reclamo_id,sub_reclamo_id,reclamo_t,comentar_t
0,1,1980,1,1,15,1501,150101,18,102,NaN,...,NaN,NaN,NaN,NaN,9,909.0,1,105,Exigen que las empresas periodísticas les otor...,NaN
1,2,1980,1,1,15,1501,150101,13,104,NaN,...,2500.0,NaN,NaN,NaN,7,712.0,1,101,Demandan una mejora en condiciones salariales.,"Exigen además: recategorización, aumento gener..."
2,3,1980,1,1,15,1501,150108,103,206,NaN,...,NaN,15.0,NaN,NaN,1,105.0,4,407,Protestan por la mala alimentación recibida en...,15 internas resultaron heridas.
3,4,1980,1,1,15,1501,150101,26,127,NaN,...,NaN,NaN,NaN,NaN,9,909.0,1,104,Protestan contra medidas de la empresa que ate...,Huelga declarada ilegal por el gobierno. Inici...
4,5,1980,1,1,8,801,80101,21,119,NaN,...,NaN,NaN,NaN,NaN,5,508.0,1,109,Denuncian el despido arbitrario de trabajadores.,También demandan la destitución de cuatro func...


**2. Definición del umbral de la violencia para la variable dependiente: "Protesta violenta"**

Criterios

*   Alguna de las acciones implica violencia según el Libro de Códigos
*   Existe alguna víctima física



In [34]:
def variable_violencia(row):
    # Códigos de Acción seleccionados por implicar violencia según el Libro de Códigos (107-114, 119, 123)
    codigo_violencia = [107, 108, 109, 110, 111, 112, 113, 114, 119, 123]

    # Extraemos los IDs mencionados de las columnas de Acción (1-4)
    acciones = [row.get('accion_1_id'), row.get('accion_2_id'),
                row.get('accion_3_id'), row.get('accion_4_id')]

    # Criterio 1 - Acción implica violencia
    es_accion_violenta = any(acc in codigo_violencia for acc in acciones)

    # Criterio 2 - Existen víctimas físicas
    muertos = pd.to_numeric(row.get('numero_muertos', 0), errors='coerce') or 0
    heridos = pd.to_numeric(row.get('numero_heridos', 0), errors='coerce') or 0
    tiene_victimas = (muertos > 0 or heridos > 0)

    if es_accion_violenta or tiene_victimas:
        return 1
    return 0

# Creamos la variable dependiente con la función
df_final['violencia_y'] = df_final.apply(variable_violencia, axis=1)

print(df_final['violencia_y'].value_counts())

violencia_y
0    21144
1     3882
Name: count, dtype: int64


**3. Revisión de la base**

In [36]:
print("Descripción del avance de la Base de Datos")
print(f"Columnas: {len(df_final.columns)}")
print(f"Observaciones: {len(df_final)}")
print("\nConteo de la Variable Dependiente (Protesta violenta):")
print(df_final['violencia_y'].value_counts())

Descripción del avance de la Base de Datos
Columnas: 27
Observaciones: 25026

Conteo de la Variable Dependiente (Protesta violenta):
violencia_y
0    21144
1     3882
Name: count, dtype: int64
